# Интерпретируемость моделей

**Проект «ИИ для учёных».** Практический блокнот к странице алгоритма «Интерпретируемость моделей».

## Что делает метод

Точная модель машинного обучения часто остаётся «чёрным ящиком»: она даёт предсказание, но не объясняет его. Методы интерпретируемости отвечают на вопрос «почему модель так решила» — какие признаки и насколько повлияли на ответ.

Это важно в науке: модель должна не только предсказывать, но и быть проверяемой. Объяснение помогает заметить, что модель опирается на артефакт данных, а не на содержательную закономерность, и согласовать выводы с предметными знаниями.

В этом блокноте мы обучим модель и применим к ней три метода интерпретации: важность признаков по перестановке, частные зависимости и локальное объяснение отдельного предсказания. Расчётное время прохождения — около пяти минут.

## Интуиция за методом

Представьте, что коллега объявляет результат своего исследования, но отказывается объяснять ход рассуждений. Вы не сможете ни проверить логику, ни заметить ошибку. Точно так же модель машинного обучения, которая лишь выдаёт предсказание, не давая объяснений, — источник недоверия, а не знания.

Методы интерпретируемости — это инструменты, которые заставляют модель «объяснить свои рассуждения». Они отвечают на три вопроса:
1. **Что важно в целом?** — какие признаки модель вообще использует для принятия решений.
2. **Как именно важно?** — растёт ли прогноз при увеличении этого признака или убывает, есть ли нелинейность.
3. **Почему именно этот прогноз?** — что именно в конкретном объекте повлияло на конкретный ответ.

**Ключевые понятия, которые встретятся в блокноте:**
- **«Чёрный ящик»** — модель, у которой сложное внутреннее устройство (сотни деревьев или слоёв), не поддающееся прямому прочтению. Градиентный бустинг и нейросети — типичные примеры.
- **Важность признаков по перестановке** — метод: признак в тестовой выборке перемешивается случайно, и измеряется, насколько упало качество. Большое падение = признак важен.
- **Частная зависимость (Partial Dependence Plot, PDP)** — график, показывающий, как меняется средний прогноз при изменении одного признака (при фиксированных остальных в среднем по выборке).
- **Локальное объяснение** — объяснение для одного конкретного объекта: какой вклад внёс каждый признак именно в этот прогноз.
- **R² (коэффициент детерминации)** — качество регрессии; 1.0 — идеальное предсказание, 0 — модель просто предсказывает среднее.
- **SHAP** — более строгий метод локальных объяснений (здесь не используется, но упоминается): основан на теории игр, распределяет вклад «честно» между признаками.

## 1. Установка библиотек

In [ ]:
%pip install -q scikit-learn numpy pandas matplotlib

## 2. Единый стиль графиков

Графики оформляются в едином визуальном стиле сайта «ИИ для учёных». Ниже встроено содержимое стилевого шаблона `notebooks/viz_style.py`.

In [ ]:
# Единый стилевой шаблон графиков проекта «ИИ для учёных».
# Значения синхронизированы с токенами темы сайта (styles/tokens.css).
VIZ_TOKENS = {
    "background": "#ffffff",
    "node_fill":  "#eef1f6",
    "node_text":  "#0e1726",
    "edge":       "#4d5e78",
    "grid":       "#dce2ec",
    "series":     ["#2563eb", "#0d9488", "#b45309", "#4d5e78"],
}
VIZ = VIZ_TOKENS


def apply_viz_style():
    """Настраивает matplotlib под единый стиль сайта."""
    import matplotlib as mpl
    from cycler import cycler
    t = VIZ_TOKENS
    mpl.rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["Segoe UI", "DejaVu Sans", "Arial", "Helvetica"],
        "font.size": 12, "axes.titlesize": 15, "axes.titleweight": "semibold",
        "axes.labelsize": 13, "xtick.labelsize": 11, "ytick.labelsize": 11,
        "legend.fontsize": 11,
        "figure.facecolor": t["background"], "axes.facecolor": t["background"],
        "savefig.facecolor": t["background"], "figure.dpi": 110, "savefig.dpi": 150,
        "axes.prop_cycle": cycler(color=t["series"]),
        "text.color": t["node_text"], "axes.labelcolor": t["node_text"],
        "axes.titlecolor": t["node_text"], "axes.edgecolor": t["edge"],
        "xtick.color": t["edge"], "ytick.color": t["edge"], "axes.linewidth": 1.2,
        "axes.grid": True, "grid.color": t["grid"], "grid.linewidth": 1.0,
        "grid.linestyle": "-", "axes.axisbelow": True,
        "lines.linewidth": 2.0, "lines.markersize": 6, "patch.linewidth": 1.5,
        "axes.spines.top": False, "axes.spines.right": False,
        "legend.frameon": False,
    })


def get_palette(n=None):
    """Возвращает список цветов рядов из токенов темы."""
    series = VIZ_TOKENS["series"]
    if n is None:
        return list(series)
    return [series[i % len(series)] for i in range(n)]


apply_viz_style()
print("Единый стиль графиков подключён.")

## 3. Демонстрационные данные

Используем встроенный в scikit-learn набор данных о стоимости жилья в Калифорнии — реальные данные, не требующие загрузки. Задача: предсказать медианную стоимость жилья в районе по его характеристикам (доход, возраст домов, плотность населения и т. д.).

Этот датасет удобен тем, что признаки содержательно интерпретируемы (доход явно должен влиять на стоимость) — это позволяет проверить, «думает» ли модель разумно.

**Что делает ячейка ниже:** загружает данные, переименовывает признаки на русский, разбивает на обучение (75 %) и проверку (25 %).

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

ds = fetch_california_housing(as_frame=True)
# Понятные русские названия признаков.
rename = {
    "MedInc": "медианный доход", "HouseAge": "возраст дома",
    "AveRooms": "комнат в среднем", "AveBedrms": "спален в среднем",
    "Population": "население", "AveOccup": "жильцов на дом",
    "Latitude": "широта", "Longitude": "долгота",
}
X = ds.data.rename(columns=rename)
y = ds.target            # медианная стоимость, сотни тыс. долларов

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=0)
print(f"Объектов: обучение {len(X_train)}, проверка {len(X_test)}")
print(f"Признаки: {list(X.columns)}")

## 4. Применение метода

### Шаг 1. Обучение модели («чёрного ящика»)

**Что делает ячейка ниже:** обучает градиентный бустинг — ансамбль из сотен деревьев решений. Это точная, но непрозрачная модель: её «внутренние веса» не читаются напрямую, как у линейной регрессии. Именно поэтому нам понадобятся методы интерпретации.

**Градиентный бустинг** — алгоритм, который строит деревья последовательно: каждое новое дерево исправляет ошибки предыдущего. `HistGradientBoostingRegressor` — быстрая версия из scikit-learn.

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import r2_score

model = HistGradientBoostingRegressor(max_iter=200, random_state=0)
model.fit(X_train, y_train)
print(f"Коэффициент детерминации R2 на проверке: "
      f"{r2_score(y_test, model.predict(X_test)):.3f}")

### Шаг 2. Важность признаков по перестановке

**Что делает ячейка ниже:** для каждого признака по очереди перемешивает его значения в тестовой выборке (разрывая связь с целевой переменной) и замеряет, насколько упал R². Среднее падение по 10 повторениям — оценка важности. Планки погрешности (усы) показывают разброс между повторениями — устойчивость оценки.

**Что мы ожидаем:** медианный доход должен быть среди самых важных признаков — это интуитивно понятно. Если лидирует что-то неожиданное (например, долгота) — стоит проверить, не артефакт ли это данных.

### Шаг 2. Важность признаков по перестановке

Метод универсален: значения одного признака случайно перемешиваются, и измеряется, насколько упало качество модели. Сильное падение означает, что модель существенно опирается на этот признак.

In [ ]:
from sklearn.inspection import permutation_importance
import matplotlib.pyplot as plt

result = permutation_importance(model, X_test, y_test, n_repeats=10,
                                random_state=0)
imp = (pd.Series(result.importances_mean, index=X.columns)
       .sort_values())

fig, ax = plt.subplots(figsize=(10, 5.6))
ax.barh(imp.index, imp.values, color=VIZ["series"][0],
        xerr=result.importances_std[np.argsort(result.importances_mean)],
        capsize=3)
ax.set_title("Важность признаков по перестановке")
ax.set_xlabel("Падение качества при перемешивании признака")
ax.set_ylabel("Признак")
fig.tight_layout()
plt.show()

**Как читать этот график.** Горизонтальные полоски — признаки, отсортированные от наименее к наиболее важному (снизу вверх). Длина полоски — насколько сильно падает R² модели, когда значения этого признака перемешиваются. Чем длиннее полоска, тем важнее признак. Усы — разброс по 10 повторениям: короткие усы означают устойчивую оценку. Признаки с длиной ≈ 0 практически не влияют на качество модели.

### Шаг 3. Частная зависимость

Важность показывает, **насколько** признак влияет — но не **как**. Частная зависимость показывает форму связи: как меняется средний прогноз при изменении одного признака, пока остальные зафиксированы на своих средних значениях.

**Что делает ячейка ниже:** строит PDP-кривую для самого важного признака. Форма кривой может быть линейной (рост/снижение), насыщающей (S-образной) или немонотонной — каждый из вариантов несёт содержательную информацию.

In [ ]:
from sklearn.inspection import PartialDependenceDisplay

top_feature = imp.index[-1]      # самый важный признак
fig, ax = plt.subplots(figsize=(10, 5.4))
PartialDependenceDisplay.from_estimator(
    model, X_test, [top_feature], ax=ax,
    line_kw={"color": VIZ["series"][1], "linewidth": 2.5})
ax.set_title(f"Частная зависимость прогноза от признака «{top_feature}»")
ax.set_xlabel(top_feature)
ax.set_ylabel("Средний прогноз стоимости")
fig.tight_layout()
plt.show()

**Как читать этот график.** Ось X — значение признака (от минимального до максимального в выборке). Ось Y — средний прогноз стоимости при данном значении признака. Кривая вверх — признак повышает прогноз; кривая вниз — снижает. Если кривая выходит на плато, увеличение признака перестаёт влиять на прогноз. Важно: это **усреднённый эффект** по всей выборке, а не для каждого конкретного района.

### Шаг 4. Локальное объяснение отдельного предсказания

Предыдущие методы описывают модель **в целом**. Для конкретного объекта полезно локальное объяснение: какой вклад внёс каждый признак именно в **этот** прогноз и в какую сторону?

**Что делает ячейка ниже:** берёт один район из тестовой выборки, сравнивает его признаки с «типичным» районом (средние значения по обучающей выборке) и для каждого признака измеряет, как изменился бы прогноз, если бы этот признак был «типичным». Разница — вклад признака.

Это упрощённый аналог SHAP-значений, реализованный без дополнительных библиотек.

In [ ]:
sample = X_test.iloc[[0]]
base = X_train.mean().to_frame().T          # «типичный» объект
base_pred = model.predict(base)[0]
full_pred = model.predict(sample)[0]

contributions = {}
for col in X.columns:
    modified = sample.copy()
    modified[col] = base[col].values        # вернуть признак к среднему
    contributions[col] = full_pred - model.predict(modified)[0]

contr = pd.Series(contributions).sort_values()

fig, ax = plt.subplots(figsize=(10, 5.6))
colors = [VIZ["series"][2] if v < 0 else VIZ["series"][0]
          for v in contr.values]
ax.barh(contr.index, contr.values, color=colors)
ax.axvline(0, color=VIZ["edge"], linewidth=1.2)
ax.set_title("Вклад признаков в прогноз для одного района")
ax.set_xlabel("Вклад в прогноз (сдвиг стоимости)")
ax.set_ylabel("Признак")
fig.tight_layout()
plt.show()

print(f"Прогноз для типичного района: {base_pred:.2f}")
print(f"Прогноз для рассматриваемого района: {full_pred:.2f}")

**Как читать этот график.** Каждая горизонтальная полоска — один признак. Длина и направление — вклад этого признака в прогноз для данного района относительно «типичного»:
- Полоска вправо (синяя) — признак **повысил** прогноз по сравнению с типичным районом.
- Полоска влево (оранжевая) — признак **снизил** прогноз.
- Длина — сила эффекта.

Сумма всех вкладов объясняет разницу между прогнозом для этого района и прогнозом для «типичного» района. Сравните с частной зависимостью: признаки, важные глобально, обычно дают крупные вклады и локально.

## 5. Интерпретация результата

- **Важность по перестановке**: длина столбца — насколько модель опирается на признак. Планки погрешности показывают устойчивость оценки. Полезная проверка: соответствует ли набор важных признаков предметным ожиданиям; неожиданный лидер может указывать на утечку данных или артефакт.
- **Частная зависимость**: форма кривой показывает характер влияния — рост, насыщение, немонотонность. Это содержательная информация о том, как модель «видит» процесс.
- **Локальное объяснение**: для конкретного объекта видно, какие признаки подняли прогноз (один цвет), а какие опустили (другой). Сумма вкладов объясняет отклонение прогноза от типичного.
- Методы согласованы: признак, важный в целом, обычно даёт крупные локальные вклады.

Помните: интерпретация объясняет поведение модели, а не саму реальность. Если модель уловила корреляцию, объяснение покажет именно эту корреляцию, а не причинную связь. Объяснение — инструмент проверки модели, а не замена причинного анализа.

## 5а. Поэкспериментируйте сами

| Что изменить | Где | Ожидаемый эффект |
|---|---|---|
| `sample = X_test.iloc[[5]]` вместо `[[0]]` | ячейка локального объяснения | Другой район — другие вклады признаков |
| `top_feature = "широта"` вместо автовыбора | ячейка PDP | Посмотрите нелинейную зависимость от географии |
| `n_repeats=30` вместо 10 | ячейка важности | Усы станут уже — более устойчивая оценка, но медленнее |
| Замените `HistGradientBoostingRegressor` на `LinearRegression` | ячейка модели | Линейная модель — другая картина важности; сравните R² |

## 6. Попробуйте на своих данных

Замените демонстрационный набор своими данными.

1. Подготовьте таблицу признаков и столбец целевой величины; разбейте на обучение и проверку.
2. Снимите комментарии в ячейке ниже и укажите свой файл.
3. Выполните ячейки раздела 4. Методы интерпретации универсальны и применимы к любой обученной модели scikit-learn (для классификации замените регрессор и метрику).

In [ ]:
# --- Шаблон загрузки своих данных ---
# raw = pd.read_csv("my_dataset.csv")
# target_col = "целевая_величина"
# X = raw.drop(columns=[target_col])
# y = raw[target_col]
# X_train, X_test, y_train, y_test = train_test_split(
#     X, y, test_size=0.25, random_state=0)
#
# model.fit(X_train, y_train)
# Далее повторите ячейки интерпретации раздела 4.

## 7. Краткие выводы

- Методы интерпретируемости не заменяют хорошую модель — сначала убедитесь, что модель работает (R² достаточно высок), только потом интерпретируйте.
- Важность по перестановке отвечает на вопрос «что важно», частная зависимость — «как именно», локальное объяснение — «почему этот конкретный прогноз».
- Три метода согласованы: признак, важный глобально, обычно даёт крупные вклады и локально; форма частной зависимости объясняет направление вклада.
- Объяснение описывает логику **модели**, а не закономерность в реальности. Важный для модели признак — зацепка для гипотезы, а не доказанная причина.
- При коррелированных признаках важность «размывается» между ними — учитывайте это при интерпретации.
- Для публикаций рассмотрите SHAP: он строже математически и поддерживает визуализацию beeswarm-диаграмм.

## Три вопроса для самопроверки

Ответьте до того, как раскроете подсказку, — это проверяет, что метод понят, а не просто запущен.

1. График важности по перестановке показывает, что «долгота» и «широта» входят в топ-3 самых важных признаков модели цен на жильё. Коллега заключает, что географическое положение является причиной высокой стоимости. Что не так с этим выводом и как правильно сформулировать то, что на самом деле показывает метод?
2. Признаки «число комнат» и «число спален» сильно коррелируют между собой (r ≈ 0.92). На графике важности по перестановке оба показывают умеренную важность. Какое явление это демонстрирует и как правильно интерпретировать совокупный вклад этих признаков?
3. Локальное объяснение для района A показывает, что «медианный доход» повысил прогноз на +1.2, а «число жильцов на дом» снизил на −0.3. Для района B «медианный доход» повысил прогноз на +0.1. Что из этого следует о применимости частной зависимости (PDP) как способа обобщить влияние дохода по всей выборке?

<details>
<summary>Показать ориентиры для ответов</summary>

1. Важность по перестановке показывает, что модель использует эти признаки для предсказания, — но не устанавливает причинно-следственную связь. Географические координаты могут быть прокси для невключённых переменных (близость к морю, транспортная доступность, школы). Правильная формулировка: «модель присваивает высокий вес географическому положению», а не «положение причинно определяет цену».
2. При высокой корреляции важность «размывается» между признаками: перемешивание одного лишь частично разрывает его связь с откликом, поскольку другой коррелят остаётся нетронутым. Суммарный вклад пары значительно выше каждого по отдельности. При интерпретации следует рассматривать коррелированные признаки как группу, а не независимо.
3. Это демонстрирует гетерогенность эффекта: влияние дохода на прогноз существенно варьируется от района к району. PDP показывает усреднённый эффект по всей выборке и скрывает эту вариацию. Для корректного анализа необходимо строить ICE-кривые (Individual Conditional Expectation) — они показывают индивидуальные зависимости для каждого наблюдения и позволяют обнаружить взаимодействия признаков.
</details>
